In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
from scipy.interpolate import splprep, splev


In [2]:
dataset_path = "/mnt/cluster/datasets/threading_il/v5/20250711_164443_034004/episode.csv"
dataset = pd.read_csv(dataset_path)


In [ ]:
dataset[["actionx", "actiony", "actionz"]] = np.nan_to_num(
        np.array(dataset[["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]].shift(-1)) - np.array(dataset[["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]]))

In [ ]:
dataset['actionx']

In [3]:
def plot_interactive_trajectory(dataset, downsample_factor=1):
    """
    Create interactive 3D plot of trajectory with specified downsampling
    
    Parameters:
    dataset (DataFrame): DataFrame containing trajectory data
    downsample_factor (int): Factor to downsample data (1 = no downsampling)
    """
    # Downsample the data
    downsampled_data = dataset.iloc[::downsample_factor]
    
    # Get the coordinates
    x = downsampled_data['relative_tip_position_x']
    y = downsampled_data['relative_tip_position_y']
    z = downsampled_data['relative_tip_position_z']
    
    # Create the 3D figure
    fig = go.Figure()
    
    # Add the main trajectory points
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=5,
            color='blue',
            opacity=0.7
        ),
        name='Trajectory Points'
    ))
    
    # Add a line connecting the points
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='lines',
        line=dict(
            color='gray',
            width=2
        ),
        opacity=0.3,
        name='Path'
    ))
    
    # Mark start and end points with labels
    fig.add_trace(go.Scatter3d(
        x=[x.iloc[0]], y=[y.iloc[0]], z=[z.iloc[0]],
        mode='markers+text',
        marker=dict(
            size=10,
            color='green',
            symbol='circle'
        ),
        text=['Start'],
        textposition='top center',
        textfont=dict(
            size=15
        ),
        name='Start'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[x.iloc[-1]], y=[y.iloc[-1]], z=[z.iloc[-1]],
        mode='markers+text',
        marker=dict(
            size=10,
            color='red',
            symbol='circle'
        ),
        text=['End'],
        textposition='top center',
        textfont=dict(
            size=15
        ),
        name='End'
    ))
    
    # Update layout for better appearance
    title = f'3D Trajectory Visualization (Downsampled by factor {downsample_factor})' if downsample_factor > 1 else '3D Trajectory Visualization'
    
    fig.update_layout(
        title=title,
        width=900,
        height=700,
        scene=dict(
            xaxis_title='X Position',
            yaxis_title='Y Position',
            zaxis_title='Z Position',
            aspectmode='cube'
        ),
        legend=dict(
            x=0.01,
            y=0.99,
            bordercolor='Black',
            borderwidth=1
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    return fig



In [9]:

dataset_path = "/mnt/cluster/datasets/threading_il/v5/20250711_164118_507201///episode.csv"
dataset = pd.read_csv(dataset_path)
fig3d = plot_interactive_trajectory(dataset, downsample_factor=6)
fig3d.show()


In [ ]:
from scipy.interpolate import splprep, splev
import numpy as np
import pandas as pd

def smooth_trajectory_multivariate(dataset, smoothing_factor=1e-4, num_points=None):
    """
    Smooths a 3D trajectory using multivariate B-spline representation with debug statements.
    
    Parameters:
        dataset (DataFrame): DataFrame with 'abs_pos_x', 'abs_pos_y', 'abs_pos_z'
        smoothing_factor (float): Smoothing parameter
        num_points (int): Number of points in output (defaults to length of input)
    
    Returns:
        DataFrame: Smoothed trajectory
    """
    if num_points is None:
        num_points = len(dataset)
    
    print(f"Dataset shape: {dataset.shape}")
    
    x = dataset['abs_pos_x'].values
    y = dataset['abs_pos_y'].values
    z = dataset['abs_pos_z'].values
    
    print(f"X range: {np.min(x)} to {np.max(x)}, length: {len(x)}")
    print(f"Y range: {np.min(y)} to {np.max(y)}, length: {len(y)}")
    print(f"Z range: {np.min(z)} to {np.max(z)}, length: {len(z)}")
    
    # Check for NaN values
    x_nans = np.isnan(x).sum()
    y_nans = np.isnan(y).sum()
    z_nans = np.isnan(z).sum()
    print(f"NaN counts - X: {x_nans}, Y: {y_nans}, Z: {z_nans}")
    
    if x_nans > 0 or y_nans > 0 or z_nans > 0:
        print("WARNING: Input data contains NaN values which will cause splprep to fail")
        # Remove NaN values for the demo
        valid_mask = ~(np.isnan(x) | np.isnan(y) | np.isnan(z))
        x = x[valid_mask]
        y = y[valid_mask]
        z = z[valid_mask]
        print(f"After removing NaNs - X: {len(x)}, Y: {len(y)}, Z: {len(z)} points remain")
    
    # Check for infinity values
    x_infs = np.isinf(x).sum()
    y_infs = np.isinf(y).sum()
    z_infs = np.isinf(z).sum()
    print(f"Infinity counts - X: {x_infs}, Y: {y_infs}, Z: {z_infs}")
    
    if x_infs > 0 or y_infs > 0 or z_infs > 0:
        print("WARNING: Input data contains infinity values which will cause splprep to fail")
        # Remove infinity values
        valid_mask = ~(np.isinf(x) | np.isinf(y) | np.isinf(z))
        x = x[valid_mask]
        y = y[valid_mask]
        z = z[valid_mask]
        print(f"After removing infinities - X: {len(x)}, Y: {len(y)}, Z: {len(z)} points remain")
    
    # Check for duplicate points
    points = np.column_stack((x, y, z))
    print(f"Total points: {len(points)}")
    
    if len(points) < 4:
        print("ERROR: Not enough valid points (minimum 4 required for cubic spline)")
        raise ValueError("Not enough valid points for spline fitting")
    
    # Check for duplicate or very close points
    if len(points) > 1:
        distances = np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1))
        min_dist = np.min(distances)
        max_dist = np.max(distances)
        zero_dists = np.sum(distances < 1e-10)
        
        print(f"Point distances - Min: {min_dist:.6e}, Max: {max_dist:.6e}")
        print(f"Points with near-zero distance: {zero_dists}")
        
        if zero_dists > 0:
            print("WARNING: Found points that are too close together")
            # Create a mask for points to keep
            mask = np.ones(len(points), dtype=bool)
            mask[1:][distances < 1e-10] = False
            
            x = x[mask]
            y = y[mask]
            z = z[mask]
            print(f"After removing duplicates - X: {len(x)}, Y: {len(y)}, Z: {len(z)} points remain")
    
    # Check for collinearity
    if len(points) > 2:
        # Calculate angles between consecutive segments
        vectors = np.diff(points, axis=0)
        # Normalize vectors
        norms = np.sqrt(np.sum(vectors**2, axis=1))
        # Avoid division by zero
        norms[norms == 0] = 1.0
        unit_vectors = vectors / norms[:, np.newaxis]
        
        # Calculate dot products between consecutive unit vectors
        dots = np.sum(unit_vectors[:-1] * unit_vectors[1:], axis=1)
        # Clip to valid range for arccos
        dots = np.clip(dots, -1.0, 1.0)
        angles = np.arccos(dots) * 180 / np.pi
        
        collinear_segments = np.sum(np.abs(angles) < 1.0)
        
        print(f"Collinear segments (angle < 1°): {collinear_segments}")
        if collinear_segments > len(points) * 0.9:
            print("WARNING: Data appears highly collinear which may cause numerical issues")
    
    print(f"Final point count for spline fitting: {len(x)}")
    print(f"Using smoothing factor: {smoothing_factor}")
    
    try:
        print("Attempting cubic spline (k=3)...")
        tck, u = splprep([x, y, z], s=smoothing_factor, k=3)
        print("Success!")
    except ValueError as e:
        print(f"Cubic spline failed with error: {str(e)}")
        try:
            print("Attempting quadratic spline (k=2)...")
            tck, u = splprep([x, y, z], s=smoothing_factor, k=2)
            print("Success with quadratic spline!")
        except ValueError as e:
            print(f"Quadratic spline failed with error: {str(e)}")
            print("Attempting linear interpolation (k=1)...")
            tck, u = splprep([x, y, z], s=0, k=1)
    
    # Evaluate the spline at equally spaced points
    u_new = np.linspace(0, 1, num_points)
    x_new, y_new, z_new = splev(u_new, tck)
    
    # Create output dataframe
    smoothed = pd.DataFrame({
        'abs_pos_x': x_new, 
        'abs_pos_y': y_new,
        'abs_pos_z': z_new
    })
    
    # Copy any other columns from original dataset
    for col in dataset.columns:
        if col not in ['abs_pos_x', 'abs_pos_y', 'abs_pos_z']:
            # For simplicity, just copy the first num_points values or pad with NaN
            if len(dataset) >= num_points:
                smoothed[col] = dataset[col].values[:num_points]
            else:
                values = list(dataset[col].values) + [np.nan] * (num_points - len(dataset))
                smoothed[col] = values
    
    return smoothed

In [ ]:
smooth_data = smooth_trajectory_multivariate(dataset)
#uniform_data = reparameterize_trajectory(smooth_data)
fig3d = plot_interactive_trajectory(smooth_data, downsample_factor=6)
fig3d.write_html('/home/mazzalore/Desktop/interactive_3d_trajectory.html')
#fig3d.show()

In [ ]:


def reparameterize_trajectory(dataset, num_points=None):
    """
    Reparameterize the trajectory to have more uniform point distribution
    
    Parameters:
    dataset (DataFrame): DataFrame containing trajectory data
    num_points (int): Number of points in output trajectory (default: same as input)
    
    Returns:
    DataFrame: New DataFrame with reparameterized points
    """
    # Get original coordinates
    x = dataset['abs_pos_x'].values
    y = dataset['abs_pos_y'].values
    z = dataset['abs_pos_z'].values
    
    points = np.column_stack((x, y, z))
    
    if num_points is None:
        num_points = len(points)
    
    # Create an empty dataframe for the result
    result = pd.DataFrame(columns=dataset.columns)
    
    # Calculate cumulative distance along the path
    cumulative_dist = np.zeros(len(points))
    for i in range(1, len(points)):
        cumulative_dist[i] = cumulative_dist[i-1] + np.linalg.norm(points[i] - points[i-1])
    
    # If trajectory doesn't move, return original
    if cumulative_dist[-1] == 0:
        return dataset
        
    # Create new equidistant points
    total_dist = cumulative_dist[-1]
    target_dists = np.linspace(0, total_dist, num_points)
    
    # Interpolate to get new points
    x_new = np.interp(target_dists, cumulative_dist, x)
    y_new = np.interp(target_dists, cumulative_dist, y)
    z_new = np.interp(target_dists, cumulative_dist, z)
    
    # Create the result dataframe
    result['abs_pos_x'] = x_new
    result['abs_pos_y'] = y_new
    result['abs_pos_z'] = z_new
    
    for col in dataset.columns:
        if col not in ['abs_pos_x', 'abs_pos_y', 'abs_pos_z']:
            result[col] = dataset[col].values
    
    return result


In [ ]:

def smooth_trajectory_spline(dataset, smoothing_factor=1e-4):
    """
    Smooths a 3D trajectory using cubic spline with shared arc-length parameterization.

    Parameters:
        dataset (DataFrame): DataFrame with 'abs_pos_x', 'abs_pos_y', 'abs_pos_z'
        smoothing_factor (float): Smoothing parameter (0 = interpolation, >0 = smoothing)
    
    Returns:
        DataFrame: Smoothed trajectory
    """
    x = dataset['abs_pos_x'].values
    y = dataset['abs_pos_y'].values
    z = dataset['abs_pos_z'].values
    points = np.column_stack((x, y, z))

    # Parameterize by cumulative arc-length
    ds = np.linalg.norm(np.diff(points, axis=0), axis=1)
    s = np.concatenate([[0], np.cumsum(ds)])
    s /= s[-1]  # normalize to [0,1]

    # Fit smoothing splines
    spl_x = UnivariateSpline(s, x, s=smoothing_factor)
    spl_y = UnivariateSpline(s, y, s=smoothing_factor)
    spl_z = UnivariateSpline(s, z, s=smoothing_factor)

    # Evaluate on same number of points
    s_eval = np.linspace(0, 1, len(dataset))
    x_smooth = spl_x(s_eval)
    y_smooth = spl_y(s_eval)
    z_smooth = spl_z(s_eval)

    smoothed = dataset.copy()
    smoothed['abs_pos_x'] = x_smooth
    smoothed['abs_pos_y'] = y_smooth
    smoothed['abs_pos_z'] = z_smooth
    return smoothed

In [ ]:

def smooth_trajectory_kalman(dataset, process_var=1e-2, measurement_var=1e-1):
    """
    Apply a Kalman filter to smooth 3D trajectory (x,y,z) with a constant velocity model.
    
    Parameters:
        dataset (DataFrame): Must contain 'abs_pos_x', 'abs_pos_y', 'abs_pos_z'
        process_var (float): Variance of the process noise
        measurement_var (float): Variance of the measurement noise

    Returns:
        DataFrame: Smoothed trajectory
    """
    dt = 1.0  # time step (can be adjusted or estimated from timestamps)
    x = dataset['abs_pos_x'].values
    y = dataset['abs_pos_y'].values
    z = dataset['abs_pos_z'].values
    n = len(x)

    # State: [x, y, z, vx, vy, vz]
    A = np.eye(6)
    A[0, 3] = A[1, 4] = A[2, 5] = dt

    H = np.zeros((3, 6))
    H[0, 0] = H[1, 1] = H[2, 2] = 1

    Q = process_var * np.eye(6)      # Process noise
    R = measurement_var * np.eye(3)  # Measurement noise

    x_est = np.zeros((n, 6))         # Estimated state
    P = np.eye(6)                    # Initial state covariance
    x_k = np.zeros(6)                # Initial state estimate

    for i in range(n):
        z_k = np.array([x[i], y[i], z[i]])

        # Predict
        x_k = A @ x_k
        P = A @ P @ A.T + Q

        # Update
        y_k = z_k - H @ x_k
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x_k = x_k + K @ y_k
        P = (np.eye(6) - K @ H) @ P

        x_est[i] = x_k

    smoothed = dataset.copy()
    smoothed['abs_pos_x'] = x_est[:, 0]
    smoothed['abs_pos_y'] = x_est[:, 1]
    smoothed['abs_pos_z'] = x_est[:, 2]
    return smoothed

In [ ]:
dataset.head()

In [ ]:
def moving_average_3d(dataset, window_size):
    """
    Apply a moving average filter to a 3D trajectory.
    
    Parameters:
    trajectory: numpy array of shape (n_points, 3) containing x, y, z coordinates
    window_size: size of the moving average window
    
    Returns:
    smoothed_trajectory: numpy array of shape (n_points, 3) with smoothed coordinates
    """
    x = dataset['abs_pos_x'].values
    y = dataset['abs_pos_y'].values
    z = dataset['abs_pos_z'].values
    
    # Create an empty dataframe for the result
    result = pd.DataFrame(columns=dataset.columns)

    # Stack to create trajectory array of shape (3, n_points)
    trajectory = np.stack((x, y, z), axis=0)
    
    n_points = trajectory.shape[1]  # Number of points is in the second dimension
    smoothed_trajectory = np.zeros_like(trajectory)
    
    # Apply moving average to each dimension
    for dim in range(3):  # For each dimension (x, y, z)
        for i in range(n_points):
            # Calculate window boundaries, handling edges
            window_start = max(0, i - window_size//2)
            window_end = min(n_points, i + window_size//2 + 1)
            
            # Apply moving average to this dimension
            window_points = trajectory[dim, window_start:window_end]
            smoothed_trajectory[dim, i] = np.mean(window_points)
            
        # Create the result dataframe
    result['abs_pos_x'] = smoothed_trajectory[0,:]
    result['abs_pos_y'] = smoothed_trajectory[1, :]
    result['abs_pos_z'] = smoothed_trajectory[2, :]
    # Copy other columns (if any)
    for col in dataset.columns:
        if col not in ['abs_pos_x', 'abs_pos_y', 'abs_pos_z']:
            result[col] = dataset[col].values
    

    return result

In [ ]:

def smooth_trajectory_sg(dataset, window_length=7, polyorder=4, mode='interp'):
    """
    Savitzky–Golay smoothing for 3D trajectories.
    window_length must be odd and >= polyorder+2.
    mode='interp' avoids edge artifacts by interpolating boundary points.
    """
    sm = dataset.copy()
    for coord in ['abs_pos_x','abs_pos_y','abs_pos_z']:
        sm[coord] = savgol_filter(
            sm[coord].values,
            window_length=window_length,
            polyorder=polyorder,
            mode=mode
        )
    return sm


In [ ]:
smooth_data = smooth_trajectory_spline(dataset)
#uniform_data = reparameterize_trajectory(smooth_data)
fig3d = plot_interactive_trajectory(smooth_data, downsample_factor=6)
fig3d.write_html('/home/mazzalore/Desktop/interactive_3d_trajectory.html')
fig3d.show()

In [ ]:
# Visualize
fig3d = plot_interactive_trajectory(dataset, downsample_factor=6)
fig3d.show()
fig3d.write_html('/home/mazzalore/Desktop/interactive_3d_trajectory.html')

In [ ]:
import chart_studio
import chart_studio.plotly 
import plotly.express as px

name = 'mazzalore'
k = 'CaprYD7jzuBAHc9OMhWJ'
chart_studio.tools.set_credentials_file(username=name, api_key=k)

chart_studio.plotly.plot(fig3d, filename='smoothed_demonstration_resampled', auto_open=True)

In [ ]:
fig3d.show()
